# Smoothing and tracking under shake

**Question.** The marker tracker blends each new position with the previous one (`SMOOTHING = 0.6`). Under camera shake, does that blending make the logged marker positions better or worse?

**Why it matters.** Plan Task 3 (`docs/plans/2026-11-16-tracker-gate.md`) proposes `SMOOTHING = 1.0`, which turns smoothing off. This study gives the evidence for that decision. The CSV columns do not change either way.

**How to use this notebook.** Run the cells from top to bottom (*Run All*). The first run detects markers in every shaken video, which takes a few minutes. Later runs read the saved detections from `output/` and finish in seconds.

| Section | What it does |
|---|---|
| 1-2 | Setup |
| 3-5 | The variable, the method and the measures |
| 6-7 | fdcl-gwu footage: detect, sweep, charts |
| 8 | Shadow-ArUco footage: the same, as a second check |
| 9 | Export the charts for the report |
| 10 | Conclusion |

## 1. Install

From the repository root, after the README's installation steps:

```bash
pip install -r analysis/requirements.txt
```

The test footage is not in git. Download it as the README's **Test Footage** section shows, into `test_footage/fdcl-gwu/` and `test_footage/shadow-aruco/`.

PNG export (section 9) needs a Chromium-based browser: Chrome, Chromium, Edge, Brave or Vivaldi. Without one, run `plotly_get_chrome` once, or use the HTML charts.

## 2. Setup

This cell finds the repository root, so the notebook works from any folder, and imports the study code:

- `study.py` shakes the footage, detects markers, replays the tracker and computes the measures.
- `charts.py` draws the Plotly charts.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
sys.path.insert(0, str(ROOT))

from analysis.smoothing import charts, study  # noqa: E402

print("repository:", ROOT)

## 3. The variable: the smoothing weight

For each confirmed marker, the tracker updates its reported position with

$$p_{\text{reported}} \leftarrow p_{\text{reported}} + w \,(p_{\text{detected}} - p_{\text{reported}})$$

- $w = 1.0$: the reported position is the detected one, so there is no smoothing.
- $w = 0.6$: the current setting. Each report keeps 40% of the previous one.
- Smaller $w$: steadier output that lags further behind.

The tracker's other rules (3 hits in 5 frames, and the move limit) compare against the **raw** detection, not the smoothed one. So $w$ should change *where* markers are reported, but not *which* markers are reported. Section 6 checks this.

## 4. Method

1. **Shake the footage.** `testing/vibration_test.py` moves each frame along a random path: slow sway, fast jitter, rotation, motion blur along the move, rolling-shutter wobble at higher levels, and a zoom that hides the black borders. The levels are below (amplitudes in px for an 800 px wide frame).
2. **Repeat with 3 random shake paths** (seeds 1, 2 and 3). Seed 1 is the project's logged baseline. Charts show the mean, with a band from the lowest to the highest seed.
3. **Detect once, replay many.** Markers are detected once per level and seed, and the detections are saved. They are then fed through a fresh tracker for each weight from 0.1 to 1.0. This is valid because $w$ does not change detection.
4. **Know the truth.** The shake is a known transform of each frame. Applying it to the clean video's detections gives where every marker really is in each shaken frame.

In [ ]:
for name, level in study.SHAKE_LEVELS.items():
    print(f"{name:<10}", level)
print("weights:", study.WEIGHTS)
print("seeds:  ", study.SEEDS)

## 5. Measures

| Measure | Meaning | Better |
|---|---|---|
| **Frame error** (px) | Distance from each reported position to the marker's true position *in that frame*. This is what a logged pixel position should match. | Lower |
| **Steady error** (px) | Distance to where the marker would be if the camera were held still (zoom only, no sway, jitter or rotation). This shows whether smoothing "removes the shake". | Lower |
| **Jitter** (px) | RMS of the reported position's step from one frame to the next. | Lower looks calmer |
| **True jitter** (px) | The same step for the true position over the same frames: how much the marker really moves in the image. | Reference |
| **Tracked** (%) | Share of the clean video's real markers that are reported. | Higher |
| **False IDs** (per 100 frames) | Reported IDs that are not real markers. | Lower |

**How the truth is built.** The true position is the detector's result on the clean video, moved by the known shake. So at the clean level, frame error is zero at $w = 1.0$ by construction, and the measure cannot reward a filter that removes the detector's own noise. When a frame holds two markers with the same ID (42 frames on fdcl-gwu, 19 on Shadow-ArUco), each report is matched to the nearest true copy.

Jitter on its own can mislead. If the marker really moves 39 px per frame, a correct output also moves 39 px per frame. Jitter below the true jitter means the output is *not following* real motion.

## 6. fdcl-gwu: detect and sweep

`fdcl-gwu` is 821 frames of 1080p video with 8 ArUco markers (`DICT_ARUCO_ORIGINAL`). It is the project's shake baseline.

In [ ]:
fdcl = study.load_or_detect("fdcl-gwu")  # a few minutes the first time, then read from output/
print(f"{len(fdcl.runs)} shaken runs, real markers {sorted(fdcl.real)}")
fdcl_summary = study.summarise(study.sweep(fdcl))

**Check: recall and false IDs do not depend on the weight.** Within each level, every weight should show the same *tracked* and *false IDs* values. *raw* is the detector alone, with no tracker.

In [ ]:
print(study.format_table(fdcl_summary))

**Key numbers** for the raw detector, the current weight (0.6) and no smoothing (1.0):

In [ ]:
KEY = ("frame_err_px", "steady_err_px", "jitter_px", "true_jitter_px")
print(study.format_table([s for s in fdcl_summary if s["weight"] in (None, 0.6, 1.0)], metrics=KEY))

## 7. fdcl-gwu: results

### 7.1 Frame error

How far the logged position is from the marker's true position in the same frame. Each line is a shake level, and the band spans the three seeds.

In [ ]:
figures = {}
figures["fdcl-frame-error"] = charts.metric_vs_weight(
    fdcl_summary, "fdcl-gwu", "frame_err_px", "fdcl-gwu: error against the true position in each frame", "Frame error (px)")
figures["fdcl-frame-error"].show()

**Reading it:** if the error falls towards $w = 1.0$, smoothing adds error. The gap between levels shows that the error grows with how fast the image moves. At severe shake the tracker reports very few markers (see *tracked* in section 6), so treat that line as a rough guide, and note its wide band.

### 7.2 Steady error

How far the logged position is from where the marker would be with a still camera. If smoothing "removed the shake", this would fall as $w$ gets smaller.

In [ ]:
figures["fdcl-steady-error"] = charts.metric_vs_weight(
    fdcl_summary, "fdcl-gwu", "steady_err_px", "fdcl-gwu: error against the still-camera position", "Steady error (px)")
figures["fdcl-steady-error"].show()

**Reading it:** two effects pull against each other.
- Smoothing removes part of the fast frame-to-frame jitter, which brings the report closer to the still-camera position.
- This video's own camera moves, by 9.5 px per frame before any shake is added (see *true jitter* at clean in section 6), and heavy smoothing lags behind that motion.

On fdcl-gwu the two balance around $w = 0.5$. Section 8 shows a still scene, where smoothing keeps helping. In both videos, most of the error that remains is slow sway (periods of 1-4 s), which no short filter removes.

### 7.3 Jitter

How much the logged position moves from one frame to the next.

In [ ]:
figures["fdcl-jitter"] = charts.metric_vs_weight(
    fdcl_summary, "fdcl-gwu", "jitter_px", "fdcl-gwu: frame-to-frame jitter of reported positions", "Jitter, RMS step per frame (px)")
figures["fdcl-jitter"].show()

**Reading it:** jitter falls steadily as $w$ drops. Compare it with *true jitter* in the key numbers table: below that value, the output is no longer following the marker's real motion in the image.

### 7.4 The trade-off at moderate shake

Each point is one weight. Down is steadier and left is more accurate. The dashed line is how much the marker truly moves per frame.

In [ ]:
figures["fdcl-tradeoff"] = charts.tradeoff(fdcl_summary, "fdcl-gwu", "moderate")
figures["fdcl-tradeoff"].show()

**Reading it:** at $w = 1.0$ the jitter equals the true motion and the error is near zero. Smaller weights buy calmer output only by moving away from where the marker actually is.

### 7.5 One marker over time

The x position of the marker seen most often, over 100 frames at moderate shake (seed 1). Gaps are frames where the tracker reports nothing. Zoom in the HTML version to see every frame.

In [ ]:
TRACE_WINDOW = (400, 500)
fdcl_trace = study.trace(fdcl, level="moderate", weights=(1.0, 0.6))
figures["fdcl-trace"] = charts.trace_chart(fdcl_trace, window=TRACE_WINDOW)
figures["fdcl-trace"].show()

## 8. Shadow-ArUco: a second check

`Shadow-ArUco video_5` is 1,334 frames at 800 × 600 (`DICT_ARUCO_MIP_36h12`), with different markers, lighting and resolution. If it shows the same pattern, the result is not a quirk of one video.

In [ ]:
shadow = study.load_or_detect("shadow-aruco")
print(f"{len(shadow.runs)} shaken runs, real markers {sorted(shadow.real)}")
shadow_summary = study.summarise(study.sweep(shadow))
print(study.format_table([s for s in shadow_summary if s["weight"] in (None, 0.6, 1.0)],
                         metrics=("tracked", "false_ids", *KEY)))

In [ ]:
for name, metric, title, y_title in [
    ("shadow-frame-error", "frame_err_px", "error against the true position in each frame", "Frame error (px)"),
    ("shadow-steady-error", "steady_err_px", "error against the still-camera position", "Steady error (px)"),
    ("shadow-jitter", "jitter_px", "frame-to-frame jitter of reported positions", "Jitter, RMS step per frame (px)"),
]:
    figures[name] = charts.metric_vs_weight(shadow_summary, "shadow-aruco", metric, f"Shadow-ArUco: {title}", y_title)
    figures[name].show()

**Reading it:** frame error behaves as on fdcl-gwu, with smoothing adding error. Steady error is different: it falls steadily as $w$ drops (moderate: 19.2 px at 1.0, 16.6 px at 0.6, 11.4 px at 0.1). This camera is still before the shake, so there is no scene motion to lag behind, and the jitter that smoothing removes is pure shake. So smoothing *does* give a steadier estimate of where the marker sits in a still view, but only by moving away from where it appears in each frame.

## 9. Export for the report

Every chart is saved to `analysis/smoothing/output/` as interactive HTML and as PNG (2× scale). That folder is not in git, so rerun this notebook to regenerate it.

In [ ]:
for name, fig in figures.items():
    for path in charts.save(fig, name):
        print(path.relative_to(ROOT))

## 10. Conclusion

Numbers are means over the three shake seeds, at moderate shake unless stated. They come from the run on 2026-11-16. If you change the code or footage, rerun the notebook and check them against the tables above.

1. **Smoothing does not change which markers are reported.** *Tracked* and *false IDs* are identical at every weight on both videos: fdcl-gwu 55.2% and 0.0 per 100 frames, Shadow-ArUco 53.6% and 0.6.
2. **Smoothing adds position error.** The error against the true position in each frame is 15.2 px at $w = 0.6$ and 1.7 px at $w = 1.0$ on fdcl-gwu, and 6.3 px against 1.4 px on Shadow-ArUco. The cost grows with shake: at severe shake on Shadow-ArUco, it is 12.6 px against 2.1 px. Severe fdcl-gwu reports too few markers (0.3%) to judge.
3. **In each frame, the calmer output is lag.** At $w = 1.0$ the jitter is close to the marker's true motion in the image: 39.4 px against 39.3 px on fdcl-gwu, and 19.1 px against 22.0 px on Shadow-ArUco. At $w = 0.6$ it drops to 23.4 px and 10.0 px, because the reports stop following where the marker appears.
4. **Against the still-camera position, smoothing helps on a still scene but not on a moving one.**
   - **Shadow-ArUco** (the camera is still before the shake): the error falls as $w$ drops, from 19.2 px at 1.0 to 16.6 px at 0.6 (−14%) and 11.4 px at 0.1 (−41%). Smoothing removes part of the fast per-frame shake.
   - **fdcl-gwu** (the camera already moves 9.5 px per frame): the error is lowest around $w = 0.5$-$0.6$ (38 px), against 42.6 px at 1.0 and 85 px at 0.1. Heavy smoothing lags behind the scene's own motion.
   - In both, most of the remaining error is slow sway, which a short filter cannot remove, and every gain costs frame error.
5. **Without shake, the effect is sub-pixel.** On clean Shadow-ArUco, $w = 0.6$ halves the jitter (0.6 px to 0.3 px). That jitter is the detector's own noise on a still scene, and smoothing probably moves the report *closer* to the real marker. The measure cannot show that (see Limits), but either way the size is well under a pixel.

**What this means for Task 3.**
- **For logged per-frame pixel positions, the evidence supports `SMOOTHING = 1.0`.** Smoothing at 0.6 makes the frame error 4-9 times larger and changes nothing about which markers are reported.
- **A steady estimate is a different need.** Something like centring the drone over a marker does gain from smoothing when the scene is still (Shadow-ArUco: −14% at 0.6). That argues for filtering downstream, where the drone's own motion is known, rather than in the log.
- The decision belongs to the project lead.

**Limits.**
- The shake is simulated, on two recorded videos. Neither is flight footage, and neither uses the contest dictionary `DICT_4X4_50`.
- The "truth" is the detector's result on the clean video, moved by the known shake. It includes the detector's own noise, so frame error cannot reward a filter that removes that noise. This biases frame error towards $w = 1.0$ by up to the detector's noise level (about 1-2 px), which is small next to the 6-15 px cost of $w = 0.6$.
- fdcl-gwu's own camera motion is part of the lag cost on that video.
- With duplicate IDs, each report is matched to the nearest true copy, which is slightly lenient (42 and 19 frames).
- It leaves out the rolling-shutter wobble (a few px, at moderate and severe only).

**Further work.**
- Repeat the study on flight footage.
- If a smoothed signal is ever needed, compare a 1€ filter (for live use) with an offline forward-backward smoother (for logs).